# 13 - Pass 7: every training label, FGM and pairs-then-FGM (one commit, GPU)

Same structure as notebooks 08, 11 and 12: fetch the code from GitHub, run the tests, check the data on the CPU, then train two runs at a time (one per GPU).

**What Pass 7 asks** (vault: Experiment Plan, "Pass 7 plan"): with **all 22,800 training labels**, plain BERT (A1, Pass 1) reaches 93.35% test accuracy on average. Do the two strongest recipes so far push that past 94%?

| Arm | What it trains on |
|---|---|
| A7 | all labels, BERT + FGM (epsilon 0.5, as in every pass since 2b) |
| A19 | stage 1: the generated pairs only (exactly Pass 4's A15, 720 steps); stage 2: all labels with FGM |
| A1, seed 0, tag `rerun` | all labels, plain BERT: must equal Pass 1's A1_p100_s0 exactly (re-run check) |

5 seeds for A7 and A19, test scored, masked test sets scored. **No setting is picked on test**: epsilon and the stage-1 length are frozen from earlier passes.

The seed paper reports 93.88% accuracy on its own 80/20 split (not ours), so that number is a reference point, not a like-for-like comparison.

**Before you run, in the right-hand panel:**
- **Input:** the same `gainsplit-processed` version as notebooks 10-12.
- **Internet: on**, **Accelerator: GPU T4 x2**.
- Then **Save Version -> Save & Run All (Commit)**. This is the longest run so far (about 2 hours); you can close the browser while it runs.


## 1. Get the code
Clones the public repo and prints the **commit**. Every run records it in its `result.json`.

In [ ]:
import os, subprocess, sys, time
T_START = time.time()
REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

## 2. Tests (about a minute)
The repo's tests train a tiny random BERT for a few steps on made-up data, including the pair arms, the pair batches and the explanation step. They prove the pieces fit together before real GPU time is spent. If anything fails, stop and paste the output.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed: see output above"

## 3. Check the data on the CPU (pre-flight)
Checks every file against `manifest.json` and `manifest_pass3.json`, then loads the `unfiltered` pair set and checks it. Expected: 44,834 rows (22,417 pairs). Any problem raises an error here, before training starts.


In [ ]:
import torch
from gainsplit.data import find_data_dir, verify, preflight
from gainsplit.plans import PLANS

PLAN = "pass7"
OUT_DIR = f"/kaggle/working/results/{PLAN}"
configs = PLANS[PLAN]()

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
DATA_DIR = find_data_dir("/kaggle/input")
verify(DATA_DIR)
print("data:", DATA_DIR, "- all files match manifest.json")
from gainsplit.plans import PASS6_PRETRAIN_STEPS
print("stage-1 steps:", PASS6_PRETRAIN_STEPS)
display(preflight(configs, DATA_DIR))

## 4. The list of runs and the time estimate
`pass7` = 11 runs at 100% labels (7,130 training steps each), longest first: A19 x5, A7 x5, A1 re-run x1.

Estimated run time (from Pass 1's A1 at 100%, 12.7 min, and the measured FGM and stage-1 costs): A7 about 21 min, A19 about 23 min, A1 about 13 min. Total about 233 run-minutes, so **about 2 hours on two GPUs**, plus about 5 minutes for steps 1-3.


In [ ]:
for c in configs:
    print(f"{c.run_id:24s} arm={c.arm:3s} labels={c.budget:3d}% seed={c.seed} eps={c.fgm_epsilon:g} "
          f"pretrain_steps={c.pretrain_steps}")
print(len(configs), "runs")


## 5. Run
One run per GPU; when a GPU frees up, the next run starts. Each run's log is in `results/<plan>/logs/<run_id>.log`. If the session stops halfway, run this cell again: finished runs are skipped.

In [ ]:
from gainsplit.runner import run_all
table = run_all(configs, DATA_DIR, OUT_DIR)
table.to_csv(f"{OUT_DIR}/summary.csv", index=False)
print(f"total so far: {(time.time() - T_START) / 60:.1f} min")

## 6. First look (the full analysis runs later on the CPU)
1. **Re-run check:** A1_p100_s0_rerun must have exactly Pass 1's A1_p100_s0 test scores (difference 0).
2. **Stage-1 check:** each A19 run's stage-1 dev macro-F1 must equal Pass 4's A15 (10%, same seed).
3. Test **accuracy** and macro-F1 (points) per arm, next to A1 at 100% (Pass 1, 93.35% accuracy). Also the cue-masked F1 and iSarcasmEval AUC.

What to look for: does A7 or A19 average 94% accuracy or more over the 5 seeds?


In [ ]:
import json
from pathlib import Path
import pandas as pd

rows = []
for f in sorted(Path(OUT_DIR).glob("runs/*/result.json")):
    r = json.loads(f.read_text()); c = r["config"]; sc = r["scores"]
    rows.append({"arm": c["arm"], "seed": c["seed"], "tag": c.get("tag", ""),
                 "stage1_dev_f1": (r.get("pretrain") or {}).get("dev", {}).get("macro_f1", float("nan")) * 100,
                 "test_acc": sc["test"]["accuracy"] * 100, "test_f1": sc["test"]["macro_f1"] * 100,
                 "cuemask_f1": sc["test_cuemask"]["macro_f1"] * 100,
                 "isarc_auc": sc["isarc_test"]["roc_auc"] * 100, "minutes": r["minutes"]})
res = pd.DataFrame(rows)
runs_dir = Path(REPO_DIR) / "results"

# 1. re-run check against Pass 1
p1 = json.loads((runs_dir / "pass1" / "runs" / "A1_p100_s0" / "result.json").read_text())["scores"]["test"]
rr = res[res["tag"] == "rerun"]
if len(rr):
    d = abs(rr["test_f1"].iloc[0] - p1["macro_f1"] * 100)
    print(f"re-run check: Pass 1 test F1 {p1['macro_f1'] * 100:.4f}, now {rr['test_f1'].iloc[0]:.4f} ->",
          "PASS" if d == 0 else f"DIFFERENCE {d:.6f} - report this first")

# 2. stage-1 check for A19 against Pass 4's A15
a15 = {json.loads(f.read_text())["config"]["seed"]: json.loads(f.read_text())["scores"]["dev"]["macro_f1"] * 100
       for f in (runs_dir / "pass4" / "runs").glob("A15_p10_s*/result.json")}
s1 = res[res["arm"] == "A19"].assign(a15_dev_f1=lambda d: d["seed"].map(a15))
if len(s1):
    print("stage-1 check:", "PASS (all equal A15)" if (s1["stage1_dev_f1"] == s1["a15_dev_f1"]).all()
          else "DIFFERENCE - report this first")

# 3. means per arm, next to A1 at 100% from Pass 1
base = pd.read_csv(runs_dir / "pass1" / "summary.csv")
base = base[(base["arm"] == "A1") & (base["budget"] == 100)]
print(f"A1 at 100% (Pass 1, 5 seeds): test accuracy {base['test_accuracy'].mean() * 100:.2f}, "
      f"macro-F1 {base['test_macro_f1'].mean() * 100:.2f}, best seed accuracy {base['test_accuracy'].max() * 100:.2f}")
main = res[res["tag"] == ""]
display(main.groupby("arm")[["test_acc", "test_f1", "cuemask_f1", "isarc_auc", "minutes"]]
        .agg(["mean", "std", "max"]).round(3))


## 7. Download
Zips the results folder. A commit keeps it in the Output tab: download `results_<plan>.zip` and send it with the log.

In [ ]:
import shutil
print(shutil.make_archive(f"/kaggle/working/results_{PLAN}", "zip", OUT_DIR))
print(f"notebook time: {(time.time() - T_START) / 60:.1f} min")